# Conformal prediction para regressao

Conformal prediction transforma uma previsao pontual em um intervalo de predicao. A ideia e reservar uma parte dos dados para calibracao, observar os erros do modelo nessa parte e usar um quantil desses erros para montar os intervalos.

Para cobertura alvo de 90%, usamos `alpha = 0.10`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


## Dataset

Escolha um dos dois arquivos da pasta:

- `diabetes.csv`, alvo `disease_progression`
- `california_housing.csv`, alvo `median_house_value`

In [ ]:
arquivo = "california_housing.csv"
alvo = "median_house_value"

# Para usar o outro dataset:
# arquivo = "diabetes.csv"
# alvo = "disease_progression"

alpha = 0.10

df = pd.read_csv(arquivo)
X = df.drop(columns=alvo)
y = df[alvo]

print(df.shape)
df.head()


## Separando treino, calibracao e teste

O modelo aprende no treino. A calibracao fica separada para estimar o tamanho do erro. O teste fica para avaliar os intervalos no final.

In [ ]:
X_treino_cal, X_teste, y_treino_cal, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42
)

X_treino, X_cal, y_treino, y_cal = train_test_split(
    X_treino_cal, y_treino_cal, test_size=0.25, random_state=42
)

print("treino:", X_treino.shape)
print("calibracao:", X_cal.shape)
print("teste:", X_teste.shape)


## Split conformal comum

No split conformal comum, todos os pontos recebem a mesma largura de intervalo:

`previsao +- q`

O valor `q` e um quantil dos erros absolutos no conjunto de calibracao.

In [ ]:
modelo = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
)

modelo.fit(X_treino, y_treino)


In [ ]:
pred_cal = modelo.predict(X_cal)
erros_cal = np.abs(y_cal - pred_cal)

n_cal = len(erros_cal)
nivel_quantil = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal
q = np.quantile(erros_cal, nivel_quantil, method="higher")

q


In [ ]:
pred_teste = modelo.predict(X_teste)

baixo_comum = pred_teste - q
alto_comum = pred_teste + q

cobertura_comum = ((y_teste >= baixo_comum) & (y_teste <= alto_comum)).mean()
largura_comum = np.mean(alto_comum - baixo_comum)
mae_comum = mean_absolute_error(y_teste, pred_teste)

pd.Series({
    "cobertura": cobertura_comum,
    "largura_media": largura_comum,
    "mae": mae_comum,
    "q": q,
})


## Split conformal com modelo de variancia

Agora a largura pode mudar de ponto para ponto. Usamos o conjunto de treino inteiro para treinar um modelo de media e, com os residuos desse modelo no proprio treino, treinamos outro modelo para prever o tamanho do erro.

Depois calibramos os erros normalizados:

`score = |y - previsao| / escala_prevista`

No teste, o intervalo fica:

`previsao +- q * escala_prevista`

In [ ]:
modelo_media = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
)

modelo_media.fit(X_treino, y_treino)


In [ ]:
pred_treino = modelo_media.predict(X_treino)
residuos = y_treino - pred_treino
variancia_observada = residuos ** 2

modelo_variancia = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)

modelo_variancia.fit(X_treino, variancia_observada)


In [ ]:
pred_cal_adapt = modelo_media.predict(X_cal)
escala_cal = np.sqrt(modelo_variancia.predict(X_cal))

scores_cal = np.abs(y_cal - pred_cal_adapt) / escala_cal

n_cal = len(scores_cal)
nivel_quantil = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal
q_adapt = np.quantile(scores_cal, nivel_quantil, method="higher")

q_adapt


In [ ]:
pred_teste_adapt = modelo_media.predict(X_teste)
escala_teste = np.sqrt(modelo_variancia.predict(X_teste))

baixo_adapt = pred_teste_adapt - q_adapt * escala_teste
alto_adapt = pred_teste_adapt + q_adapt * escala_teste

cobertura_adapt = ((y_teste >= baixo_adapt) & (y_teste <= alto_adapt)).mean()
largura_adapt = np.mean(alto_adapt - baixo_adapt)
mae_adapt = mean_absolute_error(y_teste, pred_teste_adapt)

comparacao = pd.DataFrame({
    "metodo": ["split comum", "com modelo de variancia"],
    "cobertura": [cobertura_comum, cobertura_adapt],
    "largura_media": [largura_comum, largura_adapt],
    "mae": [mae_comum, mae_adapt],
    "q": [q, q_adapt],
})

comparacao


## Intervalos no teste

No metodo comum, a largura e fixa. No metodo com variancia, a largura muda conforme a escala prevista para cada ponto.

In [ ]:
intervalos = pd.DataFrame({
    "y_real": y_teste,
    "pred_comum": pred_teste,
    "baixo_comum": baixo_comum,
    "alto_comum": alto_comum,
    "pred_variancia": pred_teste_adapt,
    "baixo_variancia": baixo_adapt,
    "alto_variancia": alto_adapt,
})

intervalos.head(10).round(2)


## Exemplo sintetico com variancia mudando

Agora vamos usar dados em que o ruido depende de `abs(x)`. Primeiro olhamos os dados puros, depois fazemos o split, depois aplicamos cada metodo separadamente.

In [ ]:
np.random.seed(42)

n = 3000
std_dev = 0.35

x = np.random.uniform(low=-1, high=1, size=n)
y = (x**3) + 2 * np.exp(-6 * (x - 0.3)**2)
y = y + np.random.normal(scale=std_dev * np.abs(x), size=n)
df_sintetico = pd.DataFrame({'x': x, 'y': y})

plt.figure(figsize=(7, 5))
plt.scatter(df_sintetico["x"], df_sintetico["y"], s=12, alpha=0.35)
plt.title("Dados puros")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

df_sintetico.head()


## Split dos dados sinteticos

Vamos separar treino, calibracao e teste. O treino ajusta o modelo, a calibracao escolhe o quantil e o teste avalia os intervalos.

In [ ]:
X_s = df_sintetico[["x"]]
y_s = df_sintetico["y"]

X_treino_cal_s, X_teste_s, y_treino_cal_s, y_teste_s = train_test_split(
    X_s, y_s, test_size=0.20, random_state=42
)

X_treino_s, X_cal_s, y_treino_s, y_cal_s = train_test_split(
    X_treino_cal_s, y_treino_cal_s, test_size=0.25, random_state=42
)

plt.figure(figsize=(7, 5))
plt.scatter(X_treino_s["x"], y_treino_s, s=12, alpha=0.25, label="treino")
plt.scatter(X_cal_s["x"], y_cal_s, s=12, alpha=0.45, label="calibracao")
plt.scatter(X_teste_s["x"], y_teste_s, s=12, alpha=0.45, label="teste")
plt.title("Split")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

print("treino:", X_treino_s.shape)
print("calibracao:", X_cal_s.shape)
print("teste:", X_teste_s.shape)


## CP comum no exemplo sintetico

O split conformal comum usa uma largura unica para todos os pontos.

In [ ]:
modelo_s = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=30,
    n_jobs=-1,
    random_state=42,
)

modelo_s.fit(X_treino_s, y_treino_s)

pred_cal_s = modelo_s.predict(X_cal_s)
erros_cal_s = np.abs(y_cal_s - pred_cal_s)

n_cal_s = len(erros_cal_s)
nivel_quantil_s = np.ceil((n_cal_s + 1) * (1 - alpha)) / n_cal_s
q_s = np.quantile(erros_cal_s, nivel_quantil_s, method="higher")

pred_teste_s = modelo_s.predict(X_teste_s)
baixo_comum_s = pred_teste_s - q_s
alto_comum_s = pred_teste_s + q_s

cobertura_comum_s = ((y_teste_s >= baixo_comum_s) & (y_teste_s <= alto_comum_s)).mean()
largura_comum_s = np.mean(alto_comum_s - baixo_comum_s)
mae_comum_s = mean_absolute_error(y_teste_s, pred_teste_s)

ordem_s = np.argsort(X_teste_s["x"].to_numpy())
x_plot_s = X_teste_s["x"].to_numpy()[ordem_s]
y_plot_s = y_teste_s.to_numpy()[ordem_s]

plt.figure(figsize=(8, 5))
plt.scatter(x_plot_s, y_plot_s, s=12, alpha=0.35, label="teste")
plt.plot(x_plot_s, pred_teste_s[ordem_s], color="black", label="previsao")
plt.fill_between(x_plot_s, baixo_comum_s[ordem_s], alto_comum_s[ordem_s], alpha=0.25, label="intervalo")
plt.title("CP comum: intervalo fixo")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

resultado_comum_s = pd.Series({
    "cobertura": cobertura_comum_s,
    "largura_media": largura_comum_s,
    "mae": mae_comum_s,
    "q": q_s,
})

resultado_comum_s


## CP com modelo de variancia no exemplo sintetico

Agora usamos os residuos no treino para estimar a variancia. A largura do intervalo passa a depender de `x`.

In [ ]:
modelo_media_s = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=30,
    n_jobs=-1,
    random_state=42,
)

modelo_media_s.fit(X_treino_s, y_treino_s)

pred_treino_s = modelo_media_s.predict(X_treino_s)
residuos_s = y_treino_s - pred_treino_s
variancia_observada_s = residuos_s ** 2

modelo_variancia_s = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=30,
    n_jobs=-1,
    random_state=42,
)

modelo_variancia_s.fit(X_treino_s, variancia_observada_s)

pred_cal_adapt_s = modelo_media_s.predict(X_cal_s)
escala_cal_s = np.sqrt(modelo_variancia_s.predict(X_cal_s))
scores_cal_s = np.abs(y_cal_s - pred_cal_adapt_s) / escala_cal_s

q_adapt_s = np.quantile(scores_cal_s, nivel_quantil_s, method="higher")

pred_teste_adapt_s = modelo_media_s.predict(X_teste_s)
escala_teste_s = np.sqrt(modelo_variancia_s.predict(X_teste_s))

baixo_adapt_s = pred_teste_adapt_s - q_adapt_s * escala_teste_s
alto_adapt_s = pred_teste_adapt_s + q_adapt_s * escala_teste_s

cobertura_adapt_s = ((y_teste_s >= baixo_adapt_s) & (y_teste_s <= alto_adapt_s)).mean()
largura_adapt_s = np.mean(alto_adapt_s - baixo_adapt_s)
mae_adapt_s = mean_absolute_error(y_teste_s, pred_teste_adapt_s)

plt.figure(figsize=(8, 5))
plt.scatter(x_plot_s, y_plot_s, s=12, alpha=0.35, label="teste")
plt.plot(x_plot_s, pred_teste_adapt_s[ordem_s], color="black", label="previsao")
plt.fill_between(x_plot_s, baixo_adapt_s[ordem_s], alto_adapt_s[ordem_s], alpha=0.25, label="intervalo")
plt.title("CP com modelo de variancia: intervalo adaptativo")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

comparacao_sintetico = pd.DataFrame({
    "metodo": ["split comum", "com modelo de variancia"],
    "cobertura": [cobertura_comum_s, cobertura_adapt_s],
    "largura_media": [largura_comum_s, largura_adapt_s],
    "mae": [mae_comum_s, mae_adapt_s],
    "q": [q_s, q_adapt_s],
})

comparacao_sintetico
